# Worm gear calibration tests: what the M#-WORM-GEAR runs say

The Speed Calibration page's **M1/M2/M3-WORM-GEAR** tests step one motor through 2 worm turns each way while sidereal
tracking holds the sky, and record each plate-solve sync as that motor's angle error (`driver/control_worm.py`). Each
result -- fit, checks, timing and every kept sample -- goes into `data/worm_profile.json`: the latest per motor under
`calibration` (what the Approve/Reject buttons act on), and the last 15 under `calibration_history`.

This notebook reads that file (read-only) and answers, in order:

0. **What is in the file?** The correction in use, the results waiting for review, the run history.
1. **Did each run measure a worm?** The sweep, the worm on its own after the backlash and drift terms are removed, and
   what the fit leaves over.
2. **Do reruns agree?** The main question: a profile is only worth applying if the same motor gives the same worm
   wherever and whenever it is measured.
3. **Does the result depend on where the mount pointed?** If reruns disagree, is it the pose?
4. **What should go into the profile, and what would it change?** All runs combined, against the profile in use.
5. **Is the model right?** A pure 6.0 deg sine, the same in both directions.
6. **How long does a test take, and what should Nina's wait be?** Settle times, discarded syncs, slack.
7. **Twin only:** do the fits recover the simulated worm, and are their error bars honest?

**The model** (`fit_worm_calibration`): at the test motor's MCU angle (517, deg) with worm phase
`phi = 360 x angle / 6.0`, the error is `err = a sin(phi) + b cos(phi) = A sin(phi + p)` (arcsec of motor angle; true angle
= MCU angle + err), fitted next to an offset per direction (backlash) and a quadratic trend in time and in angle (drift,
the other motors' worms as tracking turns them). `coef = [a, b]` is what the profile stores; `A = hypot(a, b)`;
`p = atan2(b, a)` puts the peak (+A) at `phi = 90 - p`.

**Phasors.** Section 2 onward plots each result as the point `(a, b)`. The distance between two points is the size (in arcsec)
of the worm left uncorrected if one were used to correct the other -- so the scatter of the points *is* the
correction you can expect. 1 deg of phase = 60" of motor angle along the worm; a phase off by `dp` leaves
`2 A sin(dp / 2)`: 5 deg leaves ~9 % of A, 10 deg ~17 %, 30 deg ~52 %.

Re-run top to bottom after new tests. `PROFILE` below picks the file.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
sys.path.insert(0, os.path.abspath('.'))
import importlib, control_worm
importlib.reload(control_worm)
from control_worm import (WormFeedForward, GEAR_WARNINGS, MIN_TURNS, MIN_SIGNIFICANCE, SETTLE_ARCSEC, SETTLE_HOLD_S,
                          JUMP_ARCSEC, _design, _lstsq)
from pe_analysis import phase_consistency

PROFILE = os.path.abspath(os.path.join('..', 'data', 'worm_profile.json'))
TEMPLATE = 'plotly_dark'
MOTORS = ('M1', 'M2', 'M3')
COLOURS = {'M1': 'royalblue', 'M2': 'mediumseagreen', 'M3': 'orange'}
DIR_COLOURS = {1: 'deepskyblue', -1: 'white', 0: 'grey'}
DIR_NAMES = {1: 'angle increasing', -1: 'angle decreasing', 0: 'start (not fitted)'}

with open(PROFILE) as f:
    raw = json.load(f)
ff = WormFeedForward.load(PROFILE)
WORM = float(raw.get('worm_theta', 6.0))
TRUTH = raw.get('twin_truth')                     # only in files written by the twin
print(f"{PROFILE}\nworm {WORM:g} deg, harmonics {raw.get('harmonics')}, "
      f"{len(raw.get('calibration_history', []))} runs in calibration_history"
      + (f"\nTWIN FILE: {raw.get('twin_note', '')}" if TRUTH else ''))

/home/jdm/projects/alpaca-benro-polaris/data/worm_profile.json
worm 6 deg, harmonics [1], 15 runs in calibration_history
TWIN FILE: calibration/calibration_history are twin simulations (source twin): Worm(35/60/50", 6.0 deg, h2 0.15, seed 11), 5 poses, syncs every 15 s, 2" noise


In [2]:
# ── helpers ─────────────────────────────────────────────────────────────────────────────────
def amp_phase(c):
    a, b = c
    return float(np.hypot(a, b)), float(np.degrees(np.arctan2(b, a)) % 360)

def wrap180(x):
    return (np.asarray(x, float) + 180) % 360 - 180

def samples_df(entry):
    s = pd.DataFrame(entry.get('samples', []))
    for k in ('settle_s', 'since_settle_s', 'a_ra', 'a_dec'):
        if k not in s:
            s[k] = np.nan
    s['err'] = s['err_arcsec']
    if 'theta_raw' in s:
        th = np.array(s['theta_raw'].tolist(), float)
        for m in range(3):
            s[f'theta{m + 1}'] = th[:, m]
    return s

def fit_run(s, harmonics=(1,), worm=None):
    # The test's own fit (fit_worm_calibration), with the covariance of a, b and the pieces to plot: the worm
    # alone (fit minus the backlash/drift terms) and the residuals.
    worm = worm or WORM
    k = s['direction'].to_numpy() != 0
    angle, err, t, d = (s.loc[k, c].to_numpy(float) for c in ('angle', 'err', 't', 'direction'))
    X, nw = _design(angle, t, d, worm, harmonics)
    beta, r, s2 = _lstsq(X, err)
    cov = s2 * np.linalg.pinv(X.T @ X)
    nuisance = X[:, nw:] @ beta[nw:]
    return dict(keep=k, coef=beta[:2], cov=cov[:2, :2], beta=beta, nw=nw, resid=r, rms=float(np.sqrt(s2)),
                model=X @ beta, worm_only=err - nuisance, angle=angle, d=d, t=t)

def worm_curve(c, x):
    phi = 2 * np.pi * np.asarray(x) / WORM
    return c[0] * np.sin(phi) + c[1] * np.cos(phi)

def ellipse(c, cov, k=1.0, n=60):
    # k-sigma error ellipse of a 2-D estimate
    w, v = np.linalg.eigh(np.asarray(cov, float))
    a = np.linspace(0, 2 * np.pi, n)
    pts = (v @ (np.sqrt(np.maximum(w, 0))[:, None] * np.vstack([np.cos(a), np.sin(a)]))) * k
    return c[0] + pts[0], c[1] + pts[1]

def weighted_mean(coefs, covs):
    # Inverse-variance mean of 2-D estimates; chi2/dof says whether they agree within their errors (~1) or scatter
    # more (>1: something run-dependent). The returned covariance is inflated by chi2/dof when > 1 (Birge).
    W = [np.linalg.inv(C) for C in covs]
    S = np.linalg.inv(sum(W))
    m = S @ sum(Wi @ c for Wi, c in zip(W, coefs))
    chi2 = sum(float((c - m) @ Wi @ (c - m)) for Wi, c in zip(W, coefs))
    dof = max(2 * (len(coefs) - 1), 1)
    return m, S * max(chi2 / dof, 1.0), chi2 / dof

def pooled_fit(runs_s, harmonics=(1,), worm=None):
    # One worm shared by all runs, each run keeping its own backlash offsets and drift (different nights/poses).
    worm = worm or WORM
    blocks, ys, nuis = [], [], []
    for s in runs_s:
        k = s['direction'].to_numpy() != 0
        X, nw = _design(*(s.loc[k, c].to_numpy(float) for c in ('angle', 't', 'direction')), worm, harmonics)
        blocks.append(X); ys.append(s.loc[k, 'err'].to_numpy(float)); nuis.append(X.shape[1] - nw)
    nw = 2 * len(harmonics)
    n, p = sum(len(y) for y in ys), nw + sum(nuis)
    X = np.zeros((n, p)); r0, c0 = 0, nw
    for B, q in zip(blocks, nuis):
        X[r0:r0 + len(B), :nw] = B[:, :nw]
        X[r0:r0 + len(B), c0:c0 + q] = B[:, nw:]
        r0 += len(B); c0 += q
    y = np.concatenate(ys)
    beta, r, s2 = _lstsq(X, y)
    cov = s2 * np.linalg.pinv(X.T @ X)
    return beta[:nw], cov[:nw, :nw], float(np.sqrt(s2)), r, X, y, beta

## 0. What is in the file?

**The correction in use** (`motors`): what the worm feed-forward applies when `pec_worm_ff` is on. `angle_reference` =
`zeta` means the coefficients are on MCU angles (517: set by an approved M#-WORM-GEAR test, the same every session);
otherwise they are on 518 angles (learnt from guided nights by `learn_worm.py`). M1's 518 angle carries each session's
compass heading, so **only zeta-referenced phases can be compared with the test results** below.

**Waiting for review** (`calibration`): the latest result per motor; `applied` = its row was approved.

**History** (`calibration_history`): every run, oldest first.

In [3]:
rows = []
for m in range(3):
    M = MOTORS[m]
    c = ff.coef[m][:2] if ff is not None and 1 in ff.harmonics else np.zeros(2)
    A, p = amp_phase(c)
    rows.append({'motor': M, 'a': round(c[0], 2), 'b': round(c[1], 2), 'A"': round(A, 1), 'phase': round(p, 1) if A else None,
                 'angle_reference': (ff.angle_reference.get(M) if ff else None) or raw.get('angles', '518 (theta_raw)')})
display(Markdown('**Profile in use**'))
display(pd.DataFrame(rows).set_index('motor'))
prov = {k: v for k, v in raw.items() if k not in ('motors', 'calibration', 'calibration_history', 'twin_truth', 'units')}
print('provenance:', json.dumps({k: (v if not isinstance(v, list) or len(v) < 4 else f'{len(v)} items') for k, v in prov.items()}))

review = pd.DataFrame([{'motor': M, 'created': e.get('created'), 'status': e.get('status'), 'A"': e.get('amplitude_arcsec'),
                        'phase': e.get('phase_deg'), 'applied': e.get('applied'), 'source': e.get('source', 'mount')}
                       for M, e in raw.get('calibration', {}).items()])
display(Markdown('**Waiting for review** (`calibration`)'))
display(review.set_index('motor') if len(review) else 'none')

history = raw.get('calibration_history') or [{'motor': M, **e} for M, e in raw.get('calibration', {}).items()]
runs = []
for i, e in enumerate(history):
    s = samples_df(e)
    if len(s) == 0:
        continue
    f = fit_run(s)
    timing = e.get('timing') or {}
    disc = timing.get('discarded') or {}
    pose = e.get('pose_azaltroll')
    runs.append(dict(
        run=f"{i:02d} {e['motor']} {str(e.get('created', ''))[5:16].replace('T', ' ')}", motor=e['motor'],
        created=e.get('created'), source=e.get('source', 'mount'), status=e.get('status'), n=int(f['keep'].sum()),
        A=amp_phase(f['coef'])[0], phase=amp_phase(f['coef'])[1], a=f['coef'][0], b=f['coef'][1],
        se_A=float(np.sqrt(np.trace(f['cov']) / 2)), rms=f['rms'], sens=e.get('checks', {}).get('sensitivity'),
        start=[round(x, 2) for x in s.loc[0, ['theta1', 'theta2', 'theta3']]] if 'theta1' in s else None,
        ra=float(s['a_ra'].mean()), dec=float(s['a_dec'].mean()),
        pose=f"az {pose[0]:.0f} alt {pose[1]:.0f}" if pose else '',
        minutes=float((s['t'].iloc[-1] - s['t'].iloc[0]) / 60), settle_med=timing.get('settle_median_s'),
        settle_max=timing.get('settle_max_s'), moving=disc.get('moving'), jump=disc.get('jump'),
        entry=e, s=s, fit=f))
R = pd.DataFrame(runs)
display(Markdown(f'**History**: {len(R)} runs'))
display(R[['run', 'source', 'status', 'n', 'A', 'phase', 'se_A', 'rms', 'sens', 'start', 'ra', 'dec', 'pose', 'minutes',
           'settle_med', 'settle_max', 'moving', 'jump']].round(2).set_index('run'))

**Profile in use**

,a,b,"A""",phase,angle_reference
motor,,,,,
M1,0.00,0.00,0.0,NaN,theta_raw (518)
M2,-12.99,62.74,64.1,101.7,theta_raw (518)
M3,-3.35,-63.99,64.1,267.0,theta_raw (518)


provenance: {"learnt_from": "7 items", "segments": 10, "fitted_motors": ["M2", "M3"], "angles": "theta_raw (518)", "amplitude_arcsec": {"M2": 64.1, "M3": 64.1}, "model": "shared amplitude, phase per motor", "created": "2026-10-03", "phase_deg": {"M2": 101.7, "M3": 267.0}, "twin_note": "calibration/calibration_history are twin simulations (source twin): Worm(35/60/50\", 6.0 deg, h2 0.15, seed 11), 5 poses, syncs every 15 s, 2\" noise", "worm_theta": 6.0, "harmonics": [1]}


**Waiting for review** (`calibration`)

,created,status,"A""",phase,applied,source
motor,,,,,,
M1,2026-10-04T09:19:07,COMPLETED,34.41,49.7,False,twin
M2,2026-10-04T09:19:13,COMPLETED,58.78,180.6,False,twin
M3,2026-10-04T09:19:20,COMPLETED,49.51,215.5,False,twin


**History**: 15 runs

,source,status,n,A,phase,se_A,rms,sens,start,ra,dec,pose,minutes,settle_med,settle_max,moving,jump
run,,,,,,,,,,,,,,,,,
00 M1 10-04 09:17,twin,COMPLETED,48,34.38,46.57,1.44,5.66,0.64,"[179.77, 50.0, 0.2]",21.50,-73.54,az 180 alt 50,12.00,8.4,11.8,0,0
01 M2 10-04 09:17,twin,COMPLETED,48,57.89,178.65,1.72,7.35,1.00,"[179.77, 50.0, 0.2]",21.49,-73.65,az 180 alt 50,12.00,7.6,9.2,0,0
02 M3 10-04 09:18,twin,COMPLETED,48,49.62,217.28,1.33,6.07,1.00,"[179.77, 50.0, 0.2]",21.49,-73.32,az 180 alt 50,12.00,10.0,13.8,0,0
03 M1 10-04 09:18,twin,COMPLETED,48,33.83,48.41,1.26,5.04,0.81,"[119.79, 35.14, 0.14]",2.20,-41.23,az 120 alt 35,12.00,8.4,12.0,0,0
04 M2 10-04 09:18,twin,COMPLETED,48,61.10,180.47,1.46,6.61,1.00,"[119.79, 35.14, 0.14]",2.19,-41.12,az 120 alt 35,12.00,7.4,9.8,0,0
05 M3 10-04 09:18,twin,COMPLETED,48,50.88,216.25,1.37,6.16,1.00,"[119.79, 35.14, 0.14]",2.20,-41.20,az 120 alt 35,12.50,10.0,14.8,2,0
06 M1 10-04 09:18,twin,COMPLETED,48,34.95,47.11,1.28,5.25,0.49,"[239.85, 61.87, 0.09]",19.23,-43.27,az 240 alt 62,12.00,8.2,11.2,0,0
07 M2 10-04 09:18,twin,COMPLETED,48,60.11,179.97,1.68,6.98,1.00,"[239.85, 61.87, 0.09]",19.23,-43.13,az 240 alt 62,12.00,7.4,10.2,0,0
08 M3 10-04 09:18,twin,COMPLETED,48,50.79,215.92,1.24,5.53,1.00,"[239.86, 61.87, 0.09]",19.23,-43.21,az 240 alt 62,12.50,10.0,14.2,2,0


## 1. Did each run measure a worm?

**What it answers:** whether the run's numbers mean anything before comparing them. One run per slider step.

- **Left, the sweep:** each kept sync's error in the order it was taken, with the full fit (worm + backlash offset per
  direction + drift). Blue = angle increasing, white = decreasing. The two legs should be the same wave shifted by the
  backlash; a large slow trend is drift (alignment, the other motors' worms).
- **Middle, the worm alone:** the error with the backlash and drift terms removed, folded onto one 6 deg worm turn.
  Both directions should fall on one sine (the fit, orange). Green dashed = the profile in use for this motor, only if it
  is on MCU angles; magenta dotted = the simulated worm (twin files).
- **Right, what's left:** the residuals on the same fold. Flat noise = the model fits; a wave = a harmonic or a
  different period the model doesn't have (section 5); a step between colours = direction-dependent error.

The heading lists the test's checks; ⚠ marks the ones the Speed Calibration row would warn about:
significance < 4 (amplitude / its error), < 1.5 worm turns covered, 2nd harmonic > 0.3 of the 1st, best period more than
0.2 deg from 6.0, forward and reverse phases > 30 deg apart, sensitivity < 0.3 (the motor barely moves the field).

In [4]:
def flags(e):
    ch = e.get('checks', {})
    out = [label(ch[k], WORM) for k, bad, label in GEAR_WARNINGS if ch.get(k) is not None and bad(ch[k], WORM)]
    if ch.get('significance') is not None and ch['significance'] < MIN_SIGNIFICANCE:
        out.append(f"significance {ch['significance']}")
    if ch.get('turns') is not None and ch['turns'] < MIN_TURNS:
        out.append(f"turns {ch['turns']}")
    return out

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.06,
                    subplot_titles=['the sweep: error by sync', f'worm alone, folded on {WORM:g} deg', 'residual, folded'])
SUB = [a.to_plotly_json() for a in fig.layout.annotations]
grid = np.linspace(0, WORM, 200)
spans, heads = [], []
for k, r in R.iterrows():
    s, f, e, M = r['s'], r['fit'], r['entry'], r['motor']
    first = k == len(R) - 1                                   # start on the latest run
    start = len(fig.data)
    kept = s[f['keep']].reset_index(drop=True)
    idx = np.flatnonzero(f['keep'])
    fold = np.mod(f['angle'], WORM)
    for d in (1, -1):
        sel = f['d'] == d
        lab = DIR_NAMES[d]
        fig.add_trace(go.Scatter(x=idx[sel], y=kept['err'][sel], mode='markers', marker=dict(size=6, color=DIR_COLOURS[d]),
                                 name=lab, legendgroup=lab, visible=first), row=1, col=1)
        fig.add_trace(go.Scatter(x=fold[sel], y=f['worm_only'][sel], mode='markers', marker=dict(size=6, color=DIR_COLOURS[d]),
                                 legendgroup=lab, showlegend=False, visible=first), row=1, col=2)
        fig.add_trace(go.Scatter(x=fold[sel], y=f['resid'][sel], mode='markers', marker=dict(size=6, color=DIR_COLOURS[d]),
                                 legendgroup=lab, showlegend=False, visible=first), row=1, col=3)
    fig.add_trace(go.Scatter(x=idx, y=f['model'], mode='lines', line=dict(color='orange', width=1.5),
                             name='fit (worm + backlash + drift)', visible=first), row=1, col=1)
    fig.add_trace(go.Scatter(x=grid, y=worm_curve(f['coef'], grid), mode='lines', line=dict(color='orange', width=2),
                             name='fitted worm', visible=first), row=1, col=2)
    m = MOTORS.index(M)
    if ff is not None and ff.angle_reference.get(M) == 'zeta':
        fig.add_trace(go.Scatter(x=grid, y=worm_curve(ff.coef[m][:2], grid), mode='lines',
                                 line=dict(color='mediumseagreen', dash='dash'), name='profile in use', visible=first), row=1, col=2)
    if TRUTH:
        T = TRUTH[M]
        tc = T['amplitude_arcsec'] * np.array([np.cos(np.radians(T['phase_deg'])), np.sin(np.radians(T['phase_deg']))])
        fig.add_trace(go.Scatter(x=grid, y=worm_curve(tc, grid), mode='lines', line=dict(color='magenta', dash='dot'),
                                 name='simulated worm (1st harmonic)', visible=first), row=1, col=2)
    spans.append((start, len(fig.data) - start))
    ch = e.get('checks', {})
    fl = flags(e)
    heads.append((r['run'], f"{r['run']}  {r['source']}  {r['status']}  n {r['n']}   A {r['A']:.1f}\" ± {r['se_A']:.1f}  phase {r['phase']:.1f}°   "
                  f"rms {r['rms']:.1f}\"  sensitivity {r['sens']}   pose {r['pose'] or r['start']}<br>"
                  f"2nd harm {ch.get('harmonic2_ratio')}  best period {ch.get('best_worm_theta')}°  fwd/rev phase "
                  f"{ch.get('phase_fwd_deg')}/{ch.get('phase_rev_deg')}° (split {ch.get('phase_split_deg')})  backlash {ch.get('backlash_arcsec')}\"  "
                  + ('⚠ ' + ', '.join(fl) if fl else 'checks ok')))
INFO = lambda t: dict(text=t, xref='paper', yref='paper', x=0, y=1.13, xanchor='left', yanchor='bottom', align='left',
                      showarrow=False, font=dict(family='monospace', size=12, color='#DDDDDD'))
steps = []
for (st, cnt), (lab, head) in zip(spans, heads):
    vis = [False] * len(fig.data); vis[st:st + cnt] = [True] * cnt
    steps.append(dict(method='update', label=lab, args=[{'visible': vis}, {'annotations': SUB + [INFO(head)]}]))
fig.update_layout(template=TEMPLATE, height=560, width=1500, margin=dict(t=140, l=70, r=30, b=60),
                  legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom'), annotations=SUB + [INFO(heads[-1][1])],
                  sliders=[dict(active=len(steps) - 1, steps=steps, pad=dict(t=50), currentvalue=dict(prefix='run: '))])
fig.update_xaxes(title_text='kept sync #', row=1, col=1)
fig.update_xaxes(title_text='MCU angle mod worm (deg)', row=1, col=2)
fig.update_xaxes(title_text='MCU angle mod worm (deg)', row=1, col=3)
fig.update_yaxes(title_text='motor angle error (arcsec)', col=1)
fig.show()

display(pd.DataFrame([{'run': r['run'], **{k: r['entry'].get('checks', {}).get(k) for k in
                       ('n', 'turns', 'significance', 'rms_arcsec', 'harmonic2_ratio', 'best_worm_theta', 'phase_split_deg',
                        'backlash_arcsec', 'sensitivity')}, 'warnings': ', '.join(flags(r['entry']))} for _, r in R.iterrows()]).set_index('run'))

,n,turns,significance,rms_arcsec,harmonic2_ratio,best_worm_theta,phase_split_deg,backlash_arcsec,sensitivity,warnings
run,,,,,,,,,,
00 M1 10-04 09:17,48,2.31,22.1,5.66,0.179,6.00,-5.5,-1.8,0.64,
01 M2 10-04 09:17,48,1.99,33.9,7.35,0.154,6.08,-0.1,2.6,1.00,
02 M3 10-04 09:18,48,1.73,39.0,6.07,0.148,5.94,1.2,2.4,1.00,
03 M1 10-04 09:18,48,2.27,26.1,5.04,0.152,5.86,-2.8,-5.3,0.81,
04 M2 10-04 09:18,48,1.82,42.0,6.61,0.141,6.00,2.0,-4.2,1.00,
05 M3 10-04 09:18,48,1.83,37.1,6.16,0.150,5.90,0.8,0.6,1.00,
06 M1 10-04 09:18,48,2.20,28.5,5.25,0.154,5.98,-3.6,-3.7,0.49,
07 M2 10-04 09:18,48,2.18,35.2,6.98,0.146,6.00,-1.8,-1.7,1.00,
08 M3 10-04 09:18,48,1.87,41.8,5.53,0.140,5.90,-0.8,0.1,1.00,


## 2. Do reruns agree?

**What it answers:** whether a motor's worm is a fixed property of its gear -- the same amplitude and phase on MCU
angles every run -- which is what a fixed feed-forward profile assumes.

**The plot:** one panel per motor, each run as its phasor `(a, b)` with its 1-sigma error ellipse (from the fit). The
white cross is the runs' inverse-variance mean with its ellipse; green square = the profile in use (if on MCU angles);
magenta star = the simulated worm (twin). Faint circles are constant amplitude; the angle around the origin is the phase.

**The table:**
- `A mean ± sd`, `phase mean`, `R`, `p`: amplitude and phase spread (R = 1 identical phases, ~0 random; p = the chance of
  that clustering from random phases, `pe_analysis.phase_consistency`);
- `chi2/dof`: the runs' scatter against their own error bars. ~1 = they agree within noise, so more runs simply average
  down; >> 1 = something differs between runs (pose, load, drift) beyond noise -- section 3 looks for it;
- `spread"`: rms distance of the runs from their mean = the worm you'd leave by applying one run instead of the mean;
  `worst"`: the largest. Compare with the amplitude: correcting is worth it while these are well below A.

Only `COMPLETED` runs count.

In [5]:
G = R[R['status'] == 'COMPLETED']
consensus, rows = {}, []
fig = make_subplots(rows=1, cols=3, subplot_titles=[f'{M} phasors (a, b)' for M in MOTORS], horizontal_spacing=0.06)
for m, M in enumerate(MOTORS):
    g = G[G['motor'] == M]
    lim = 10.0
    for _, r in g.iterrows():
        c = np.array([r['a'], r['b']])
        ex, ey = ellipse(c, r['fit']['cov'])
        fig.add_trace(go.Scatter(x=ex, y=ey, mode='lines', line=dict(color=COLOURS[M], width=1), opacity=0.5,
                                 showlegend=False, hoverinfo='skip'), row=1, col=m + 1)
        fig.add_trace(go.Scatter(x=[c[0]], y=[c[1]], mode='markers+text', marker=dict(size=8, color=COLOURS[M]),
                                 text=[r['run'][:2]], textposition='top center', textfont=dict(size=9),
                                 hovertext=f"{r['run']}<br>A {r['A']:.1f}\" phase {r['phase']:.1f}° {r['pose']}",
                                 showlegend=False), row=1, col=m + 1)
        lim = max(lim, np.abs(c).max() + 3 * r['se_A'])
    if len(g):
        coefs = [np.array([a, b]) for a, b in zip(g['a'], g['b'])]
        covs = [f['cov'] for f in g['fit']]
        mean, cov, chi = weighted_mean(coefs, covs) if len(g) > 1 else (coefs[0], covs[0], np.nan)
        consensus[M] = (mean, cov)
        ex, ey = ellipse(mean, cov)
        fig.add_trace(go.Scatter(x=ex, y=ey, mode='lines', line=dict(color='white', width=2), showlegend=False), row=1, col=m + 1)
        fig.add_trace(go.Scatter(x=[mean[0]], y=[mean[1]], mode='markers', marker=dict(symbol='x', size=12, color='white'),
                                 name='mean of runs', showlegend=m == 0), row=1, col=m + 1)
        d = np.array([np.hypot(*(c - mean)) for c in coefs])
        pc = phase_consistency(g['phase'])
        rows.append({'motor': M, 'runs': len(g), 'A mean': g['A'].mean(), 'A sd': g['A'].std(ddof=1) if len(g) > 1 else np.nan,
                     'phase mean': pc['mean'], 'phase sd': float(np.degrees(np.sqrt(-2 * np.log(max(pc['R'], 1e-9))))),
                     'R': pc['R'], 'p': pc['p'], 'mean A"': amp_phase(mean)[0], 'mean phase': amp_phase(mean)[1],
                     'mean ±"': float(np.sqrt(np.trace(cov) / 2)), 'chi2/dof': chi, 'spread"': float(np.sqrt(np.mean(d ** 2))),
                     'worst"': float(d.max())})
    if ff is not None and ff.angle_reference.get(M) == 'zeta':
        c = ff.coef[m][:2]
        fig.add_trace(go.Scatter(x=[c[0]], y=[c[1]], mode='markers', marker=dict(symbol='square', size=10, color='mediumseagreen'),
                                 name='profile in use', showlegend=True), row=1, col=m + 1)
    if TRUTH:
        T = TRUTH[M]
        tc = T['amplitude_arcsec'] * np.array([np.cos(np.radians(T['phase_deg'])), np.sin(np.radians(T['phase_deg']))])
        fig.add_trace(go.Scatter(x=[tc[0]], y=[tc[1]], mode='markers', marker=dict(symbol='star', size=14, color='magenta'),
                                 name='simulated worm', showlegend=m == 0), row=1, col=m + 1)
        lim = max(lim, np.abs(tc).max() + 5)
    a = np.linspace(0, 2 * np.pi, 120)
    for rad in np.arange(10, lim * 1.5, 10 if lim < 60 else 20):
        fig.add_trace(go.Scatter(x=rad * np.cos(a), y=rad * np.sin(a), mode='lines', line=dict(color='#444', width=0.5),
                                 showlegend=False, hoverinfo='skip'), row=1, col=m + 1)
    ax = dict(range=[-lim * 1.1, lim * 1.1], zeroline=True, zerolinecolor='#666')
    fig.update_xaxes(title_text='a = sin coefficient (")', row=1, col=m + 1, **ax)
    fig.update_yaxes(title_text='b = cos coefficient (")' if m == 0 else None, row=1, col=m + 1,
                     scaleanchor=f'x{m + 1 if m else ""}', scaleratio=1, **ax)
fig.update_layout(template=TEMPLATE, height=520, width=1500, legend=dict(orientation='h', y=1.08, x=0))
fig.show()
AGREE = pd.DataFrame(rows).set_index('motor') if rows else pd.DataFrame()
display(AGREE.round(2))

,runs,A mean,A sd,phase mean,phase sd,R,p,"mean A""",mean phase,"mean ±""",chi2/dof,"spread""","worst"""
motor,,,,,,,,,,,,,
M1,5,34.45,0.42,47.81,1.12,1.0,0.01,34.48,47.90,0.56,0.25,0.77,1.09
M2,5,59.50,1.23,179.86,0.70,1.0,0.01,59.64,179.94,0.72,0.39,1.33,2.19
M3,5,50.04,0.73,215.75,1.14,1.0,0.01,50.04,215.72,0.60,0.50,1.18,1.78


## 3. Does the result depend on where the mount pointed?

**What it answers:** if section 2 shows more scatter than noise (`chi2/dof` >> 1), whether it follows the pose -- e.g.
gravity loading the gear differently with altitude, or a low sensitivity (the motor barely moves the field) making a
run noisy.

**The plot:** pick the motor with the slider. Each point is one of its runs (labelled with the run number), as its
amplitude (top) and phase (bottom) minus the mean of that motor's runs, with the run's own 1-sigma, against the starting
MCU angle of each motor (the pose) and the sensitivity. The dashed line is a straight-line fit; the grey band is the
mean's own uncertainty.

**The table:** each slope divided by its standard error (`t`). |t| < 2 = no trend (the differences are noise); a clear
trend shows |t| > 3 consistently with a sensible cause (e.g. the M2 angle ~ altitude). With a handful of runs, one
outlying run can make a slope -- check the labels.

In [6]:
XS = {'theta1': 'start M1 angle (deg)', 'theta2': 'start M2 angle (deg)', 'theta3': 'start M3 angle (deg)', 'sens': 'sensitivity'}
YS = (('dA', 'A - mean A (")', 'seA'), ('dphase', 'phase - mean phase (deg)', 'sePh'))
rel = []
for _, r in G.iterrows():
    mean, cov = consensus[r['motor']]
    A0, p0 = amp_phase(mean)
    rel.append({'run': r['run'], 'label': r['run'][:2], 'motor': r['motor'], 'dA': r['A'] - A0,
                'dphase': float(wrap180(r['phase'] - p0)), 'seA': r['se_A'], 'sePh': float(np.degrees(r['se_A'] / max(r['A'], 1e-9))),
                'band_A': float(np.sqrt(np.trace(cov) / 2)), 'band_ph': float(np.degrees(np.sqrt(np.trace(cov) / 2) / max(A0, 1e-9))),
                'sens': r['sens'], **({f'theta{k + 1}': r['start'][k] for k in range(3)} if r['start'] else {})})
rel = pd.DataFrame(rel)
xs = [x for x in XS if x in rel and rel[x].notna().any()]
fig = make_subplots(rows=2, cols=len(xs), shared_yaxes='rows', horizontal_spacing=0.03, vertical_spacing=0.12)
spans, trend = [], []
for M in MOTORS:
    g = rel[rel['motor'] == M]
    start = len(fig.data)
    first = M == MOTORS[0]
    for i, (y, yname, se) in enumerate(YS):
        band = g['band_A' if y == 'dA' else 'band_ph'].iloc[0] if len(g) else 0
        for j, x in enumerate(xs):
            xv, yv, w = g[x].to_numpy(float), g[y].to_numpy(float), g[se].to_numpy(float)
            lo, hi = (np.nanmin(xv), np.nanmax(xv)) if len(g) else (0, 1)
            pad = (hi - lo) * 0.08 or 0.05
            fig.add_trace(go.Scatter(x=[lo - pad, hi + pad, hi + pad, lo - pad], y=[-band, -band, band, band], fill='toself',
                                     mode='none', fillcolor='rgba(160,160,160,0.18)', showlegend=False, hoverinfo='skip',
                                     visible=first), row=i + 1, col=j + 1)
            fig.add_trace(go.Scatter(x=xv, y=yv, mode='markers+text', text=g['label'], textposition='middle right',
                                     textfont=dict(size=10, color='#BBBBBB'), marker=dict(size=10, color=COLOURS[M]),
                                     error_y=dict(type='data', array=w, color=COLOURS[M], thickness=1),
                                     hovertext=g['run'], showlegend=False, visible=first), row=i + 1, col=j + 1)
            t = np.nan
            if len(g) >= 3 and np.ptp(xv) > 0:
                Xd = np.column_stack([np.ones_like(xv), xv - xv.mean()])
                Wd = 1 / np.maximum(w, 1e-9) ** 2
                cov_b = np.linalg.inv(Xd.T @ (Wd[:, None] * Xd))
                beta = cov_b @ Xd.T @ (Wd * yv)
                res = yv - Xd @ beta
                s2 = max(float(res @ (Wd * res)) / max(len(xv) - 2, 1), 1.0)    # scale up when the scatter exceeds the bars
                t = beta[1] / np.sqrt(cov_b[1, 1] * s2)
                xx = np.array([lo, hi])
                fig.add_trace(go.Scatter(x=xx, y=beta[0] + beta[1] * (xx - xv.mean()), mode='lines',
                                         line=dict(color=COLOURS[M], dash='dash', width=1.5), showlegend=False,
                                         hovertext=f't = {t:.1f}', visible=first), row=i + 1, col=j + 1)
            trend.append({'motor': M, 'vs': XS[x], y: t})
            if i == 1:
                fig.update_xaxes(title_text=XS[x], row=2, col=j + 1)
        fig.update_yaxes(title_text=yname, row=i + 1, col=1)
    spans.append((start, len(fig.data) - start))
steps = []
for M, (st, cnt) in zip(MOTORS, spans):
    vis = [False] * len(fig.data); vis[st:st + cnt] = [True] * cnt
    steps.append(dict(method='update', label=M, args=[{'visible': vis}, {f'{a}{k}.autorange': True for a in ('xaxis', 'yaxis')
                                                                          for k in [''] + list(range(2, 2 * len(xs) + 1))}]))
fig.update_layout(template=TEMPLATE, height=650, width=1500, margin=dict(t=60),
                  title="each run against its motor's mean (error bars: the run's 1-sigma; band: the mean's)",
                  sliders=[dict(active=0, steps=steps, pad=dict(t=50), currentvalue=dict(prefix='motor: '))])
fig.show()
T3 = pd.DataFrame(trend).groupby(['motor', 'vs'], sort=False).first()
display(Markdown('**Trend significance** (slope / its standard error)'))
display(T3.rename(columns={'dA': 't amplitude', 'dphase': 't phase'}).round(1))

**Trend significance** (slope / its standard error)

t amplitude  t phase
motor vs                                        
M1    start M1 angle (deg)          0.2      0.7
      start M2 angle (deg)          0.3     -0.2
      start M3 angle (deg)         -0.1     -0.9
      sensitivity                  -0.3      0.3
M2    start M1 angle (deg)         -0.6      0.2
      start M2 angle (deg)         -0.4     -0.1
      start M3 angle (deg)          0.2     -0.5
      sensitivity                   NaN      NaN
M3    start M1 angle (deg)         -0.0      0.4
      start M2 angle (deg)          0.4      0.9
      start M3 angle (deg)          0.5      1.1
      sensitivity                   NaN      NaN

## 4. What should go into the profile, and what would it change?

**What it answers:** which coefficients to apply and how much that changes the correction. Approving a test row
applies **that run** (the latest per motor). Combining runs is better when they agree (section 2), so here, per motor:

- `latest`: what approving the row would apply;
- `mean of runs`: the inverse-variance mean from section 2;
- `pooled fit`: one worm fitted to all runs' samples together, each run keeping its own backlash offsets and drift --
  the best estimate when the runs agree; its `±` is the expected error of the correction it would give;
- `in use`: the profile's current coefficients (only comparable when on MCU angles).

`vs in use"` = how much the correction would change; `vs pooled"` = how far that row is from the pooled estimate (for
`latest`: the extra worm you'd leave by approving one run instead of combining).

In [7]:
rows = []
for m, M in enumerate(MOTORS):
    g = G[G['motor'] == M]
    if not len(g):
        continue
    pc, pcov, prms, *_ = pooled_fit(list(g['s']))
    in_use = ff.coef[m][:2] if ff is not None and ff.angle_reference.get(M) == 'zeta' else None
    latest = g.iloc[-1]
    cands = [('latest ' + latest['run'][:2], np.array([latest['a'], latest['b']]), latest['se_A']),
             ('mean of runs', consensus[M][0], float(np.sqrt(np.trace(consensus[M][1]) / 2))),
             ('pooled fit', pc, float(np.sqrt(np.trace(pcov) / 2)))]
    if in_use is not None:
        cands.append(('in use', in_use, np.nan))
    if TRUTH:
        T = TRUTH[M]
        cands.append(('simulated worm', T['amplitude_arcsec'] * np.array([np.cos(np.radians(T['phase_deg'])),
                                                                          np.sin(np.radians(T['phase_deg']))]), 0.0))
    for name, c, se in cands:
        A, p = amp_phase(c)
        rows.append({'motor': M, 'estimate': name, 'a': c[0], 'b': c[1], 'A"': A, 'phase': p, '±"': se,
                     'vs in use"': float(np.hypot(*(c - in_use))) if in_use is not None else np.nan,
                     'vs pooled"': float(np.hypot(*(c - pc)))})
display(pd.DataFrame(rows).set_index(['motor', 'estimate']).round(2))

a      b     A"   phase    ±"  vs in use"  \
motor estimate                                                        
M1    latest 12       22.25  26.24  34.41   49.71  1.13         NaN   
      mean of runs    23.12  25.58  34.48   47.90  0.56         NaN   
      pooled fit      23.18  25.54  34.49   47.77  0.56         NaN   
      simulated worm  24.18  25.30  35.00   46.30  0.00         NaN   
M2    latest 13      -58.78  -0.62  58.78  180.61  1.68         NaN   
      mean of runs   -59.64   0.07  59.64  179.94  0.72         NaN   
      pooled fit     -59.57   0.10  59.57  179.90  0.71         NaN   
      simulated worm -60.00   0.31  60.00  179.70  0.00         NaN   
M3    latest 14      -40.31 -28.74  49.51  215.48  1.40         NaN   
      mean of runs   -40.62 -29.22  50.04  215.72  0.60         NaN   
      pooled fit     -40.60 -29.23  50.03  215.75  0.59         NaN   
      simulated worm -40.19 -29.74  50.00  216.50  0.00         NaN   

                      vs pooled"  
motor estimate                    
M1    latest 12             1.16  
      mean of runs          0.08  
      pooled fit            0.00  
      simulated worm        1.03  
M2    latest 13             1.07  
      mean of runs          0.08  
      pooled fit            0.00  
      simulated worm        0.48  
M3    latest 14             0.57  
      mean of runs          0.03  
      pooled fit            0.00  
      simulated worm        0.65

## 5. Is the model right?

**What it answers:** whether a pure 6.0 deg sine, the same in both directions, describes the gear -- or whether the
profile needs a 2nd harmonic, a different period, or a direction-dependent part.

- **Top, folded:** every completed run of the motor, with its own backlash and drift removed (the pooled fit), folded
  onto one worm turn, with the binned mean (white) and the pooled 1st-harmonic fit (orange). A systematic gap between
  them = shape the profile doesn't have; the 2nd-harmonic fit is printed.
- **Bottom, period scan:** the pooled fit's residual rms against an assumed worm period. A minimum at 6.0 confirms the
  period (60 teeth, 960 = 16 x 60).
- **Table:** per run, the phase fitted from each direction alone and the backlash offset. Phases that differ by more
  than their noise in a consistent direction = the gear flanks differ (the profile would need a part per direction).

In [8]:
PERIODS = np.arange(5.0, 7.5001, 0.02)
fig = make_subplots(rows=2, cols=3, subplot_titles=[f'{M}: all runs, worm alone' for M in MOTORS] +
                    [f'{M}: residual rms vs worm period' for M in MOTORS], vertical_spacing=0.15, horizontal_spacing=0.06)
h2rows = []
for m, M in enumerate(MOTORS):
    g = G[G['motor'] == M]
    if not len(g):
        continue
    pc, pcov, prms, r, X, y, beta = pooled_fit(list(g['s']))
    worm_part = X[:, :2] @ beta[:2]
    alone = worm_part + r
    ang = np.concatenate([s.loc[s['direction'] != 0, 'angle'].to_numpy(float) for s in g['s']])
    dirs = np.concatenate([s.loc[s['direction'] != 0, 'direction'].to_numpy(int) for s in g['s']])
    fold = np.mod(ang, WORM)
    for d in (1, -1):
        sel = dirs == d
        fig.add_trace(go.Scatter(x=fold[sel], y=alone[sel], mode='markers', marker=dict(size=4, color=DIR_COLOURS[d], opacity=0.5),
                                 name=DIR_NAMES[d], legendgroup=DIR_NAMES[d], showlegend=m == 0), row=1, col=m + 1)
    bins = np.linspace(0, WORM, 13)
    b = np.digitize(fold, bins) - 1
    centre = (bins[:-1] + bins[1:]) / 2
    means = [alone[b == k].mean() if (b == k).any() else np.nan for k in range(12)]
    fig.add_trace(go.Scatter(x=centre, y=means, mode='lines+markers', line=dict(color='white', width=2),
                             name='binned mean (0.5 deg)', showlegend=m == 0), row=1, col=m + 1)
    grid = np.linspace(0, WORM, 200)
    fig.add_trace(go.Scatter(x=grid, y=worm_curve(pc, grid), mode='lines', line=dict(color='orange', width=2),
                             name='pooled 1st harmonic', showlegend=m == 0), row=1, col=m + 1)
    c2, cov2, *_ = pooled_fit(list(g['s']), harmonics=(1, 2))
    h2rows.append({'motor': M, 'A1"': np.hypot(*c2[:2]), 'A2"': np.hypot(*c2[2:4]), 'A2 ±"': float(np.sqrt(np.trace(cov2[2:4, 2:4]) / 2)),
                   'A2/A1': np.hypot(*c2[2:4]) / max(np.hypot(*c2[:2]), 1e-9), 'pooled rms"': prms})
    rms = [pooled_fit(list(g['s']), worm=P)[2] for P in PERIODS]
    fig.add_trace(go.Scatter(x=PERIODS, y=rms, mode='lines', line=dict(color=COLOURS[M]), showlegend=False), row=2, col=m + 1)
    fig.add_vline(x=6.0, line=dict(color='#888', dash='dot'), row=2, col=m + 1)
    fig.update_xaxes(title_text='MCU angle mod worm (deg)', row=1, col=m + 1)
    fig.update_xaxes(title_text='assumed worm period (deg)', row=2, col=m + 1)
fig.update_yaxes(title_text='motor angle error (")', row=1, col=1)
fig.update_yaxes(title_text='residual rms (")', row=2, col=1)
fig.update_layout(template=TEMPLATE, height=820, width=1500, legend=dict(orientation='h', y=1.06, x=0))
fig.show()
display(Markdown('**2nd harmonic** (pooled fit with harmonics 1 and 2)'))
display(pd.DataFrame(h2rows).set_index('motor').round(2))
display(Markdown('**Direction** (per run)'))
display(pd.DataFrame([{'run': r['run'], 'phase fwd': r['entry']['checks'].get('phase_fwd_deg'),
                       'phase rev': r['entry']['checks'].get('phase_rev_deg'), 'split': r['entry']['checks'].get('phase_split_deg'),
                       'backlash"': r['entry']['checks'].get('backlash_arcsec')} for _, r in G.iterrows()]).set_index('run'))

**2nd harmonic** (pooled fit with harmonics 1 and 2)

,"A1""","A2""","A2 ±""",A2/A1,"pooled rms"""
motor,,,,,
M1,34.67,5.26,0.32,0.15,5.13
M2,59.93,8.75,0.23,0.15,6.92
M3,49.63,7.24,0.23,0.15,5.85


**Direction** (per run)

,phase fwd,phase rev,split,"backlash"""
run,,,,
00 M1 10-04 09:17,49.2,43.7,-5.5,-1.8
01 M2 10-04 09:17,179.5,179.4,-0.1,2.6
02 M3 10-04 09:18,216.6,217.8,1.2,2.4
03 M1 10-04 09:18,50.4,47.6,-2.8,-5.3
04 M2 10-04 09:18,179.8,181.8,2.0,-4.2
05 M3 10-04 09:18,215.8,216.6,0.8,0.6
06 M1 10-04 09:18,49.0,45.4,-3.6,-3.7
07 M2 10-04 09:18,180.9,179.1,-1.8,-1.7
08 M3 10-04 09:18,216.6,215.8,-0.8,0.1


## 6. How long does a test take, and what should Nina's wait be?

**What it answers:** how to run the test as fast as it can go without bad samples. The driver keeps a sync only once
the step has settled (within 10" of its target for 1 s: `SETTLE_ARCSEC`, `SETTLE_HOLD_S`) and then steps at once, so the test takes one solve per
position when Nina's wait between solves is at least the settle time.

- **Left, settle time** of each step (step -> settled), by position. Reversals (at +6 and -6 deg) take up backlash and
  usually settle slowest.
- **Right, slack** (settled -> the sync arrived) = Nina's wait + exposure + solve - settle time. Every kept sync's
  exposure started at least `exposure + solve` before it arrived, so if the slack is always well above your exposure +
  solve time, the wait can be shortened by the difference.
- **Table:** per run, duration, settle median / 95th percentile / max, slack, discarded syncs (`moving` = arrived
  before settling: the wait is too short; `jump` = the error jumped > 300" (`JUMP_ARCSEC`) from the last sample: an exposure that caught
  the move). The recommendation is the wait that covers 95 % of settles plus 1 s.

In [9]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['settle time by position', 'slack: settled -> sync'], horizontal_spacing=0.08)
rows = []
for _, r in R.iterrows():
    s = r['s']
    st, sl = s['settle_s'].to_numpy(float), s['since_settle_s'].to_numpy(float)
    if np.all(np.isnan(st)):
        continue
    rev = np.r_[False, np.diff(s['direction'].to_numpy()) != 0] & (s['direction'].to_numpy() != 0)
    fig.add_trace(go.Scatter(x=np.arange(len(st)), y=st, mode='lines+markers', marker=dict(size=4, color=COLOURS[r['motor']]),
                             line=dict(width=1, color=COLOURS[r['motor']]), name=r['run'], legendgroup=r['run']), row=1, col=1)
    fig.add_trace(go.Scatter(x=np.flatnonzero(rev), y=st[rev], mode='markers', marker=dict(size=11, symbol='circle-open', color='red'),
                             name='reversal', legendgroup='rev', showlegend=bool(rev.any()) and not any(t.name == 'reversal' for t in fig.data)), row=1, col=1)
    fig.add_trace(go.Box(y=sl, name=r['run'][:2] + ' ' + r['motor'], marker_color=COLOURS[r['motor']], legendgroup=r['run'],
                         showlegend=False, boxpoints='all', jitter=0.4, pointpos=0), row=1, col=2)
    v = st[~np.isnan(st)]
    gaps = np.diff(s['t'].to_numpy(float))
    rows.append({'run': r['run'], 'minutes': r['minutes'], 'positions': len(s), 'sync interval s': float(np.median(gaps)) if len(gaps) else np.nan,
                 'settle med': np.median(v), 'settle p95': np.percentile(v, 95), 'settle max': v.max(),
                 'slack min': np.nanmin(sl), 'slack med': np.nanmedian(sl), 'moving': r['moving'], 'jump': r['jump']})
fig.update_xaxes(title_text='position #', row=1, col=1)
fig.update_yaxes(title_text='seconds', row=1, col=1)
fig.update_yaxes(title_text='seconds', row=1, col=2)
fig.update_layout(template=TEMPLATE, height=520, width=1500)
fig.show()
T6 = pd.DataFrame(rows).set_index('run') if rows else pd.DataFrame()
display(T6.round(1))
if len(T6):
    wait = float(np.ceil(T6['settle p95'].max() + 1))
    print(f"Nina wait between solves: >= {wait:.0f} s covers 95 % of settles in every run (slowest step {T6['settle max'].max():.1f} s); "
          f"median slack {T6['slack med'].median():.1f} s -- the wait can be shortened by that minus your exposure + solve time.\n"
          f"Test length at that wait: ~{49 * (wait + 5) / 60:.0f} min with 5 s exposure + solve (49 positions).")

,minutes,positions,sync interval s,settle med,settle p95,settle max,slack min,slack med,moving,jump
run,,,,,,,,,,
00 M1 10-04 09:17,12.0,49,15.0,8.4,9.5,11.8,3.2,6.6,0,0
01 M2 10-04 09:17,12.0,49,15.0,7.6,8.7,9.2,5.8,7.4,0,0
02 M3 10-04 09:18,12.0,49,15.0,10.0,11.7,13.8,1.2,5.0,0,0
03 M1 10-04 09:18,12.0,49,15.0,8.4,9.5,12.0,3.0,6.6,0,0
04 M2 10-04 09:18,12.0,49,15.0,7.4,8.5,9.8,5.2,7.6,0,0
05 M3 10-04 09:18,12.5,49,15.0,10.0,11.5,14.8,3.4,5.0,2,0
06 M1 10-04 09:18,12.0,49,15.0,8.2,9.4,11.2,3.8,6.8,0,0
07 M2 10-04 09:18,12.0,49,15.0,7.4,8.4,10.2,4.8,7.6,0,0
08 M3 10-04 09:18,12.5,49,15.0,10.0,11.5,14.2,3.4,5.0,2,0


Nina wait between solves: >= 13 s covers 95 % of settles in every run (slowest step 15.2 s); median slack 6.6 s -- the wait can be shortened by that minus your exposure + solve time.
Test length at that wait: ~15 min with 5 s exposure + solve (49 positions).


## 7. Twin only: do the fits recover the simulated worm, and are their error bars honest?

**What it answers:** whether the test and its fit work, before trusting them on the mount. For files written by the
digital twin (`twin_truth`): each run's error against the simulated worm's 1st harmonic, and that error divided by
the run's own 1-sigma (`pull`). Pulls mostly within ±2 (rms ~1) = honest error bars; larger = the fit is biased or
too optimistic. Note the twin's worm has a 2nd harmonic (`h2_ratio`) that a 1st-harmonic fit leaves in the residuals.

In [10]:
if TRUTH:
    rows = []
    for _, r in G.iterrows():
        T = TRUTH[r['motor']]
        tc = T['amplitude_arcsec'] * np.array([np.cos(np.radians(T['phase_deg'])), np.sin(np.radians(T['phase_deg']))])
        d = np.array([r['a'], r['b']]) - tc
        rows.append({'run': r['run'], 'pose': r['pose'], 'A"': r['A'], 'true A"': T['amplitude_arcsec'], 'phase': r['phase'],
                     'true phase': T['phase_deg'], 'dphase': float(wrap180(r['phase'] - T['phase_deg'])), 'error"': float(np.hypot(*d)),
                     '1-sigma"': r['se_A'], 'pull': float(np.hypot(*d) / max(r['se_A'], 1e-9) / np.sqrt(2))})
    P = pd.DataFrame(rows).set_index('run')
    display(P.round(2))
    print(f"rms pull {np.sqrt((P['pull'] ** 2).mean()):.2f} (1 = error bars honest); median error {P['error\"'].median():.1f}\"")
else:
    print('not a twin file: nothing to check against')

,pose,"A""","true A""",phase,true phase,dphase,"error""","1-sigma""",pull
run,,,,,,,,,
00 M1 10-04 09:17,az 180 alt 50,34.38,35.0,46.57,46.3,0.27,0.64,1.44,0.31
01 M2 10-04 09:17,az 180 alt 50,57.89,60.0,178.65,179.7,-1.05,2.37,1.72,0.97
02 M3 10-04 09:18,az 180 alt 50,49.62,50.0,217.28,216.5,0.78,0.78,1.33,0.41
03 M1 10-04 09:18,az 120 alt 35,33.83,35.0,48.41,46.3,2.11,1.72,1.26,0.96
04 M2 10-04 09:18,az 120 alt 35,61.10,60.0,180.47,179.7,0.77,1.37,1.46,0.66
05 M3 10-04 09:18,az 120 alt 35,50.88,50.0,216.25,216.5,-0.25,0.90,1.37,0.47
06 M1 10-04 09:18,az 240 alt 62,34.95,35.0,47.11,46.3,0.81,0.50,1.28,0.28
07 M2 10-04 09:18,az 240 alt 62,60.11,60.0,179.97,179.7,0.27,0.31,1.68,0.13
08 M3 10-04 09:18,az 240 alt 62,50.79,50.0,215.92,216.5,-0.58,0.94,1.24,0.54


rms pull 0.70 (1 = error bars honest); median error 0.9"
